# Build the countries shapefile

Generate a shapefile with all the countries.


### Input data:

**World Bank**: [Official Boundaries - Admin 0](https://datacatalogfiles.worldbank.org/ddh-published/0038272/5/DR0095371/World%20Bank%20Official%20Boundaries%20(Shapefiles)/World%20Bank%20Official%20Boundaries%20-%20Admin%200.zip)

### Input data license:

Creative Commons Attribution 4.0

Giulia Cigna - giulia.cigna@polito.it<br>
Romain Thomas - romain.thomas@polito.it<br>
2026

In [1]:
import os
from dotenv import load_dotenv
import geopandas as gpd
import logging
import pandas as pd
from shapely.geometry import box
from shapely.geometry import MultiPolygon
from shapely.ops import unary_union
import matplotlib.pyplot as plt
import pycountry
import chardet
from pathlib import Path

## LOGGING

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True
)

## SETTINGS

In [3]:
if not os.path.exists(".env"):
    raise ValueError("You must create the '.env' file and set the values before running this notebook.")

load_dotenv()

True

## OUTPUT SETTINGS

In [4]:
# Get full shapefile path from environment
countries_from_wb_path = os.getenv("COUNTRIES_FROM_WB_PATH")
if countries_from_wb_path is None:
    raise ValueError("COUNTRIES_FROM_WB_PATH environment variable is not set")

# Create directory if it doesn't exist
Path(countries_from_wb_path).parent.mkdir(parents=True, exist_ok=True)

logging.info(f"Output path: {countries_from_wb_path}")

2026-03-13 17:59:26 - root - INFO - Output path: data/countries_from_wb/countries_from_wb.shp


## INPUT FILES

In [5]:
# Countries shapefile
wb_admin_countries_path = os.getenv("WB_ADMIN_COUNTRIES_PATH")
if wb_admin_countries_path is None:
    raise ValueError("WB_ADMIN_COUNTRIES_PATH environment variable is not set")

# CSV with regions ID
codes_id_path = os.getenv('CODES_ID_PATH')
if codes_id_path is None:
    raise ValueError("CODES_ID_PATH environment variable is not set")

## READING INPUT FILES

In [6]:
logging.info(f"Reading shapefile for EUROSTAT from {wb_admin_countries_path}")
gdf_countries = gpd.read_file(wb_admin_countries_path)

2026-03-13 17:59:26 - root - INFO - Reading shapefile for EUROSTAT from data/World Bank Official Boundaries - Admin 0/WB_GAD_ADM0.shp


In [7]:
logging.info(f"Reading ids from {codes_id_path}")
with open(codes_id_path, "rb") as f:
    result = chardet.detect(f.read())

# from https://pandas.pydata.org/pandas-docs/stable/user_guide/io.html#na-values
na_vals = ['-1.#IND', '1.#QNAN', '1.#IND', '-1.#QNAN', '#N/A N/A', '#N/A', 'N/A', 'n/a', 'NA', '<NA>', '#NA', 'NULL', 'null', 'NaN', '-NaN', 'nan', '-nan', 'None', '']
# avoids errors with country code "NA":
na_vals.remove('NA')

codes_id = pd.read_csv(
    codes_id_path,
    encoding=result["encoding"],
    sep=None,
    engine="python",
    keep_default_na=False,
    na_values=na_vals
)


2026-03-13 17:59:27 - root - INFO - Reading ids from ../data/codes_id.csv


## FILTER COLUMS OF INPUT FILES

In [8]:
# Scheme definition of the final shape file

schema = {
    "ID": "str:10",
    "NAME": "str:50",
    "ISO3_CODE": "str:3",
    "ISO2_CODE": "str:2",
    "ISON_CODE": "int",
    "NUM_ID": "int",
    'WB_A3': "str:3",
    'HASC_0': "str:3",
    'WB_REGION': "str:50",
    'WB_STATUS': "str:50",
    'SOVEREIGN': "str:50",
    "SOURCE": "str:50",
    "geometry": "MultiPolygon"
}


Mapping between original shape file columns and finale scheme

In [9]:
# Countries
attr_countries = [c for c in gdf_countries.columns if c != "geometry"]

logging.info(f"Attributes in the Countries shape file are: {attr_countries}")

2026-03-13 17:59:27 - root - INFO - Attributes in the Countries shape file are: ['ISO_A3', 'ISO_A2', 'WB_A3', 'HASC_0', 'GAUL_0', 'WB_REGION', 'WB_STATUS', 'SOVEREIGN', 'NAM_0']


In [10]:
# Mapping scheme specific of WORLD BANK data
countries_columns_map = {
    'NAM_0': "NAME",
    "ISO_A3": "ISO3_CODE",
    "ISO_A2": "ISO2_CODE",
    'WB_A3': "WB_A3",
    'WB_REGION': "WB_REGION",
    'WB_STATUS': "WB_STATUS",
    'SOVEREIGN': 'SOVEREIGN'
}


In [11]:
# Mapping function

def prepare_gdf(gdf, col_map, fixed_values=None):
    gdf = gdf.copy()

    # Rename columns
    gdf = gdf.rename(columns=col_map)

    # Create missing columns with placeholders
    for col in schema:
        if col not in gdf.columns and col != "geometry":
            gdf[col] = None   # placeholder

    # Apply fixed values
    if fixed_values:
        for col, val in fixed_values.items():
            gdf[col] = val

    # Keep only target columns (ORDERED by schema)
    gdf = gdf[list(schema)]

    return gdf


In [12]:
# Mapping the input gdf

gdf_countries = prepare_gdf(
    gdf_countries,
    countries_columns_map
)

logging.info(f"Attributes in the modified Countries shape file are: {gdf_countries.columns}")


2026-03-13 17:59:27 - root - INFO - Attributes in the modified Countries shape file are: Index(['ID', 'NAME', 'ISO3_CODE', 'ISO2_CODE', 'ISON_CODE', 'NUM_ID', 'WB_A3',
       'HASC_0', 'WB_REGION', 'WB_STATUS', 'SOVEREIGN', 'SOURCE', 'geometry'],
      dtype='str')


## METADATA

Fill empty metadata

In [13]:
# Dissolve to merge not unique rows
gdf_countries = gdf_countries.dissolve(by='ISO3_CODE', as_index=False)

In [14]:
# Adding IDs to identify all the regions (even if they don't have ISO codes)

# Build lookup from codes_id.csv
lookup = (
    codes_id.dropna(subset=["ISO3_CODE"])
    .drop_duplicates(subset=["ISO3_CODE"])
    .set_index("ISO3_CODE")["ID"]
)

gdf_countries["ID"] = gdf_countries["ISO3_CODE"].map(lookup)

In [15]:
# All the countries are sourced from WB
gdf_countries["SOURCE"] = "World Bank Official Boundaries"


In [16]:
# Functions to assign other ISO codes from: Countries (ISO 3166-1)
def iso3_to_num(code):
    country = pycountry.countries.get(alpha_3=code)
    return country.numeric if country else None

# Assign codes
gdf_countries["ISON_CODE"] = gdf_countries["ISO3_CODE"].apply(iso3_to_num)

In [17]:
# Add the NUM_ID from the csv file
lookup = codes_id.set_index("ID")["NUM_ID"]
gdf_countries["NUM_ID"] = gdf_countries["ID"].map(lookup)
gdf_countries["NUM_ID"] = gdf_countries["NUM_ID"].astype(str).str.strip()
gdf_countries["NUM_ID"] = pd.to_numeric(gdf_countries["NUM_ID"], errors='coerce').astype('Int64')

In [18]:
gdf_countries


,ISO3_CODE,geometry,ID,NAME,ISO2_CODE,ISON_CODE,NUM_ID,WB_A3,HASC_0,WB_REGION,WB_STATUS,SOVEREIGN,SOURCE
0,ABW,"POLYGON ((-70.05108 12.5654, -70.04885 12.5669...",AW,Aruba (Neth.),AW,533,17,ABW,AW,Other,Territory,NLD,World Bank Official Boundaries
1,AFG,"POLYGON ((70.04663 37.5436, 70.05191 37.54516,...",AF,Afghanistan,AF,004,8,AFG,AF,SAR,Member State,AFG,World Bank Official Boundaries
2,AGO,"MULTIPOLYGON (((11.77964 -17.26511, 11.77675 -...",AO,Angola,AO,024,5,AGO,AO,AFR,Member State,AGO,World Bank Official Boundaries
3,AIA,"MULTIPOLYGON (((-63.02113 18.25462, -63.01493 ...",AI,Anguilla (U.K.),AI,660,10,AIA,AI,Other,Territory,GBR,World Bank Official Boundaries
4,ALA,"MULTIPOLYGON (((19.89995 60.02297, 19.90591 60...",AX,Aaland (Fin.),AX,248,278,FIN,FI,Other,Territory,FIN,World Bank Official Boundaries
...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,XKX,"POLYGON ((21.21285 42.11033, 21.21448 42.11253...",XK,Kosovo,XK,NaN,280,XKX,XK,ECA,Member State,XKX,World Bank Official Boundaries
240,YEM,"MULTIPOLYGON (((42.16181 15.01285, 42.15954 15...",YE,Republic of Yemen,RY,887,225,YEM,YE,MENA,Member State,YEM,World Bank Official Boundaries
241,ZAF,"MULTIPOLYGON (((18.35967 -33.80697, 18.35827 -...",ZA,South Africa,ZA,710,226,ZAF,ZA,AFR,Member State,ZAF,World Bank Official Boundaries
242,ZMB,"POLYGON ((29.04742 -13.40319, 29.05138 -13.400...",ZM,Zambia,ZM,894,228,ZMB,ZM,AFR,Member State,ZMB,World Bank Official Boundaries


## SAVING OUTPUT FILE

In [19]:
# Save GeoDataFrame
gdf_countries.to_file(countries_from_wb_path)

logging.info(f"Saved Shapefile to: {countries_from_wb_path}")

2026-03-13 17:59:44 - pyogrio._io - INFO - Created 244 records
2026-03-13 17:59:44 - root - INFO - Saved Shapefile to: data/countries_from_wb/countries_from_wb.shp
